<a href="https://colab.research.google.com/github/JoThePOkeMOn/Computational-Optimization-and-Applications-Minor/blob/main/Micro_Project_1_Campus_City_Emergency_Supply_Distribution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import pulp

# --- 1. LOAD DATA ---
df_demands = pd.read_csv('/content/data/demands.csv')
df_warehouses = pd.read_csv('/content/data/warehouses.csv')
df_transport = pd.read_csv('/content/data/transportation_costs.csv')
df_facilities = pd.read_csv('/content/data/facilities.csv')

# --- 2. PROCESS DATA ---
facilities = df_demands['facility_id'].tolist()

#Create a dictionary to map IDs to real names for the final report
facility_names = {
    row['facility_id']: row['facility_name']
    for _, row in df_facilities.iterrows()
}

#Map warehouse IDs to their real names too!
warehouse_names = {
    row['warehouse_id']: row['warehouse_name']
    for _, row in df_warehouses.iterrows()
}

annual_demand = {row['facility_id']: row['daily_demand'] * 365 for _, row in df_demands.iterrows()}

warehouses = df_warehouses['warehouse_id'].tolist()
annual_capacity = {row['warehouse_id']: row['capacity'] * 365 for _, row in df_warehouses.iterrows()}
annual_fixed_cost = {row['warehouse_id']: row['construction_cost'] / 10 for _, row in df_warehouses.iterrows()}
annual_op_cost = {row['warehouse_id']: row['operational_cost'] * 365 for _, row in df_warehouses.iterrows()}

trans_cost = {w: {f: 0.0 for f in facilities} for w in warehouses}
for _, row in df_transport.iterrows():
    w = row['from_warehouse']
    f = row['to_facility']
    if w in warehouses and f in facilities:
        trans_cost[w][f] = row['cost_per_unit']

budget = 1500000

# --- 3. MODEL FORMULATION ---
model = pulp.LpProblem("Campus_City_Logistics", pulp.LpMinimize)

y = pulp.LpVariable.dicts("Build", warehouses, cat=pulp.LpBinary)
x = pulp.LpVariable.dicts("Ship", [(w, f) for w in warehouses for f in facilities], lowBound=0, cat=pulp.LpContinuous)

model += (
    pulp.lpSum([(annual_fixed_cost[w] + annual_op_cost[w]) * y[w] for w in warehouses]) +
    pulp.lpSum([trans_cost[w][f] * x[(w, f)] for w in warehouses for f in facilities]),
    "Total_Annual_Cost"
)

model += pulp.lpSum([y[w] for w in warehouses]) == 2, "Exact_Two_Warehouses"

for f in facilities:
    model += pulp.lpSum([x[(w, f)] for w in warehouses]) == annual_demand[f], f"Demand_{f}"

for w in warehouses:
    model += pulp.lpSum([x[(w, f)] for f in facilities]) <= annual_capacity[w] * y[w], f"Capacity_{w}"

model += (
    pulp.lpSum([(annual_fixed_cost[w] + annual_op_cost[w]) * y[w] for w in warehouses]) +
    pulp.lpSum([trans_cost[w][f] * x[(w, f)] for w in warehouses for f in facilities]) <= budget
), "Budget_Limit"

# --- 4. SOLVE & OUTPUT ---
model.solve()

print(f"Optimization Status: {pulp.LpStatus[model.status]}")
print(f"Total Annual Cost: ${pulp.value(model.objective):,.2f}\n")

print("--- Warehouses Selected to Build ---")
for w in warehouses:
    if y[w].varValue == 1.0:
        # Using our new dictionary here to print the real name!
        print(f"-> {warehouse_names[w]} ({w})")

print("\n--- Annual Shipping Plan ---")
for w in warehouses:
    for f in facilities:
        shipped_amount = x[(w, f)].varValue
        if shipped_amount and shipped_amount > 0:
            # Using our dictionaries here to print the human-readable route!
            print(f"Ship {shipped_amount:,.0f} units from {warehouse_names[w]} to {facility_names[f]}")

Optimization Status: Optimal
Total Annual Cost: $1,136,652.15

--- Warehouses Selected to Build ---
-> South Campus Warehouse (WH_SOUTH)
-> West Gate Warehouse (WH_WEST)

--- Annual Shipping Plan ---
Ship 9,855 units from South Campus Warehouse to Science Hall
Ship 17,520 units from South Campus Warehouse to South Dormitory
Ship 9,125 units from South Campus Warehouse to University Stadium
Ship 13,870 units from South Campus Warehouse to Community Center
Ship 12,045 units from South Campus Warehouse to Local Public School
Ship 28,835 units from South Campus Warehouse to Community Hospital
Ship 29,565 units from West Gate Warehouse to Campus Medical Center
Ship 11,680 units from West Gate Warehouse to Engineering Building
Ship 17,520 units from West Gate Warehouse to North Dormitory
Ship 9,855 units from West Gate Warehouse to Main Library
